In [1]:
import sys
from pathlib import Path

# Add the project root (parent of notebooks/) to Python's import path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [3]:
# notebooks/03a_diagnose_qa_splitter.ipynb (single cell)
import pandas as pd
from src.data_loading.loaders import load_earnings_sample

from src.preprocessing.cleaners import (
    basic_clean, strip_participant_header, QA_BOUNDARY_MARKER
)

raw = load_earnings_sample(sample_size=100, seed=42)
rows = []
for idx, r in raw.iterrows():
    cleaned = basic_clean(r["transcript"])
    cleaned, _ = strip_participant_header(cleaned)

    # Find every occurrence of the marker
    positions = []
    start = 0
    while True:
        pos = cleaned.find(QA_BOUNDARY_MARKER, start)
        if pos == -1:
            break
        positions.append(pos)
        start = pos + len(QA_BOUNDARY_MARKER)

    rows.append({
        "idx": idx,
        "ticker": r.get("ticker"),
        "total_len": len(cleaned),
        "n_occurrences": len(positions),
        "first_pos": positions[0] if positions else None,
        "last_pos": positions[-1] if positions else None,
        # Under CURRENT splitter, prepared would be:
        "current_prepared_len": positions[0] if positions else None,
        # Under FIXED splitter (skip early occurrences), prepared would be:
        "fixed_prepared_len": next(
            (p for p in positions if p >= 2000),
            positions[-1] if positions else None,
        ),
    })

diag = pd.DataFrame(rows)
print("Total calls with at least one marker:", diag["n_occurrences"].gt(0).sum())
print("Calls with 2+ occurrences:", diag["n_occurrences"].ge(2).sum())
print("Calls where current splitter drops prepared (<500):",
      (diag["current_prepared_len"] < 500).sum())
print("Calls where fixed splitter would keep prepared (>=500):",
      (diag["fixed_prepared_len"] >= 500).sum())
print()
print("Sample of calls where current vs fixed differ:")
diff = diag[diag["current_prepared_len"] != diag["fixed_prepared_len"]]
print(diff[["idx", "ticker", "n_occurrences", "first_pos",
            "last_pos", "current_prepared_len", "fixed_prepared_len"]].head(10))

Loading S&P 500 earnings transcripts split=train...
Full dataset size: 20681 transcripts
Sampled 100 transcripts
Total calls with at least one marker: 74
Calls with 2+ occurrences: 55
Calls where current splitter drops prepared (<500): 51
Calls where fixed splitter would keep prepared (>=500): 71

Sample of calls where current vs fixed differ:
    idx ticker  n_occurrences  first_pos  last_pos  current_prepared_len  \
0     0    HII              2      127.0   11257.0                 127.0   
1     1    USB              3      309.0   41281.0                 309.0   
2     2    KMB              2      200.0    2837.0                 200.0   
4     4    BSX              0        NaN       NaN                   NaN   
5     5     GM              2      266.0   17150.0                 266.0   
6     6   TSCO              0        NaN       NaN                   NaN   
7     7    APH              0        NaN       NaN                   NaN   
8     8    BMY              0        NaN      

What the four headline numbers tell us:

- "Total calls with at least one marker: 74"
    - This is the baseline. Of the 100 sampled calls, 74 contain [Operator Instructions] somewhere in the text. Those are the calls that have any chance of being split into prepared + qa. The remaining 26 calls have no marker at all and become "full" units. This matches your existing Phase 2 numbers exactly (74 qa + 26 full) — sanity check passed.

- "Calls with 2+ occurrences: 55"
    - This is the key finding. 55 out of 74 calls contain the marker twice or more. That's about three-quarters of them. The hypothesis I proposed earlier was that [Operator Instructions] appears in two structural places — once in the operator's opening (the procedural mute/unmute speech) and once at the Q&A transition. The data says yes, this is the dominant pattern, not an edge case. So the current splitter, which uses text.find() (which returns the first occurrence), is landing on the wrong marker for 55 out of 74 calls.

- "Calls where current splitter drops prepared (<500): 51"
    - This reconciles the headline anomaly. You expected 74 prepared_remarks units but saw 23. The difference is 51, and here we have 51 calls where the current splitter produced a "prepared" piece shorter than 500 chars, which then got filtered out. The mystery is solved. 74 − 51 = 23. That's your missing prepared_remarks count, accounted for exactly. Not a coincidence, not a partial explanation — a complete one.

- "Calls where fixed splitter would keep prepared (>=500): 71"
    - This is the forward-looking number. If you apply the fix (skip early occurrences, find the marker that comes after at least 2K chars of real content), 71 of the 74 splittable calls would produce a real prepared_remarks unit. That's a jump from 23 → 71. The three calls that still wouldn't produce a prepared unit even with the fix are probably either very short calls or unusual transcripts where the marker really does appear early in legitimate content small enough not to worry about now.

In [ ]:
# notebooks/03a_diagnose_qa_splitter.ipynb (single cell)
import pandas as pd
from src.data_loading.loaders import load_earnings_sample

from src.preprocessing.cleaners import (
    basic_clean, strip_participant_header, QA_BOUNDARY_MARKER
)

raw = load_earnings_sample(sample_size=100, seed=42)
rows = []
for idx, r in raw.iterrows():
    cleaned = basic_clean(r["transcript"])
    cleaned, _ = strip_participant_header(cleaned)

    # Find every occurrence of the marker
    positions = []
    start = 0
    while True:
        pos = cleaned.find(QA_BOUNDARY_MARKER, start)
        if pos == -1:
            break
        positions.append(pos)
        start = pos + len(QA_BOUNDARY_MARKER)

    rows.append({
        "idx": idx,
        "ticker": r.get("ticker"),
        "total_len": len(cleaned),
        "n_occurrences": len(positions),
        "first_pos": positions[0] if positions else None,
        "last_pos": positions[-1] if positions else None,
        # Under CURRENT splitter, prepared would be:
        "current_prepared_len": positions[0] if positions else None,
        # Under FIXED splitter (skip early occurrences), prepared would be:
        "fixed_prepared_len": next(
            (p for p in positions if p >= 2000),
            positions[-1] if positions else None,
        ),
    })

diag = pd.DataFrame(rows)
print("Total calls with at least one marker:", diag["n_occurrences"].gt(0).sum())
print("Calls with 2+ occurrences:", diag["n_occurrences"].ge(2).sum())
print("Calls where current splitter drops prepared (<500):",
      (diag["current_prepared_len"] < 500).sum())
print("Calls where fixed splitter would keep prepared (>=500):",
      (diag["fixed_prepared_len"] >= 500).sum())
print()
print("Sample of calls where current vs fixed differ:")
diff = diag[diag["current_prepared_len"] != diag["fixed_prepared_len"]]
print(diff[["idx", "ticker", "n_occurrences", "first_pos",
            "last_pos", "current_prepared_len", "fixed_prepared_len"]].head(10))

Loading S&P 500 earnings transcripts split=train...
Full dataset size: 20681 transcripts
Sampled 100 transcripts
Total calls with at least one marker: 73
Calls with 2+ occurrences: 47
Calls where current splitter drops prepared (<500): 55
Calls where fixed splitter would keep prepared (>=500): 60

Sample of calls where current vs fixed differ:
    idx ticker  n_occurrences  first_pos  last_pos  current_prepared_len  \
0     0    HII              2      127.0   11257.0                 127.0   
1     1    USB              3      309.0   41281.0                 309.0   
2     2    KMB              2      200.0    2837.0                 200.0   
4     4    BSX              0        NaN       NaN                   NaN   
5     5     GM              2      266.0   17150.0                 266.0   
6     6   TSCO              0        NaN       NaN                   NaN   
7     7    APH              0        NaN       NaN                   NaN   
8     8    BMY              0        NaN      

: 

How this makes sense
```
That shifts every marker position. A marker that sat at position 2,050 under flat cleaning might sit at 
position 1,980 under newline-preserving cleaning (or vice versa), pushing it across the 2000-char threshold in either direction. 
```
Example if old basic_clean (flat whitespace) collapsis \n\n runs into single spaces how A marker that sat at position 2,050 under flat cleaning might sit at 
position 1,980 with new basic_clean that preserves newline. If we preserve newline it should push the marker away further because with old approach you 
remove \n\n with single whitespace.

- Below is the output after running "03c_reconcile_prepared_count"
```
Loading S&P 500 earnings transcripts split=train...
Full dataset size: 20681 transcripts
Sampled 100 transcripts
Calls split (was_split=True): 73
Prepared units that survive (>=500): 60
Prepared sections in the 1-499 gap (dropped): 13

Calls split but prepared dropped: 13
    idx ticker  header_stripped  prepared_len   qa_len
9     9   TROW            False         166.0  50956.0
15   15    ELV             True         101.0  37020.0
21   21    UNH             True          13.0  47576.0
39   39   TMUS             True          24.0  65247.0
40   40    CAT             True          13.0  38505.0
45   45   MDLZ             True          24.0  33280.0
51   51    ROP             True          13.0  27312.0
59   59    LMT             True          13.0  33462.0
60   60   INTC            False         277.0  45058.0
64   64    VST             True          24.0  34032.0
69   69    EOG             True          24.0  26413.0
92   92    MRK             True          13.0  36686.0
93   93    ALL             True          24.0  33579.0
```


- I also re-ran old diagnostic code again and below is the output I now get
```
Loading S&P 500 earnings transcripts split=train...
Full dataset size: 20681 transcripts
Sampled 100 transcripts
Total calls with at least one marker: 73
Calls with 2+ occurrences: 47
Calls where current splitter drops prepared (<500): 55
Calls where fixed splitter would keep prepared (>=500): 60

Sample of calls where current vs fixed differ:
    idx ticker  n_occurrences  first_pos  last_pos  current_prepared_len  \
0     0    HII              2      127.0   11257.0                 127.0   
1     1    USB              3      309.0   41281.0                 309.0   
2     2    KMB              2      200.0    2837.0                 200.0   
4     4    BSX              0        NaN       NaN                   NaN   
5     5     GM              2      266.0   17150.0                 266.0   
6     6   TSCO              0        NaN       NaN                   NaN   
7     7    APH              0        NaN       NaN                   NaN   
8     8    BMY              0        NaN       NaN                   NaN   
10   10    KHC              2      255.0    2968.0                 255.0   
11   11   ABBV              0        NaN       NaN                   NaN   

    fixed_prepared_len  
0              11257.0  
1              10378.0  
2               2837.0  
4                  NaN  
5              17150.0  
6                  NaN  
7                  NaN  
8                  NaN  
```
It does have the stats changed now, so what should I do that move back to old cleaner with old basic_clean or what?

- I did switch to "preserve_newlines=True" please review the relevent section of attached code and let me know if I applied it correctly or not
```
def normalize_whitespace(text: str, preserve_newlines: bool = False) -> str:
    """
    Collapse whitespace runs. If preserve_newlines=True, newlines are kept
    as line boundaries (collapsing only spaces/tabs within lines and
    deduplicating consecutive blank lines). Used for transcripts where
    speaker turns need to be detectable downstream.
    """
    if not isinstance(text, str):
        return ""
    if preserve_newlines:
        # Collapse spaces/tabs within each line, strip per-line
        lines = [re.sub(r"[ \t]+", " ", ln).strip() for ln in text.split("\n")]
        # Drop empty lines but keep newlines as separators between content
        non_empty = [ln for ln in lines if ln]
        return "\n".join(non_empty)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def remove_control_chars(text: str) -> str:
...


def basic_clean(text: str, preserve_newlines: bool = False) -> str:
    """Apply minimal cleaning suitable for both corpora."""
    text = remove_control_chars(text)
    text = normalize_whitespace(text, preserve_newlines=preserve_newlines)
    return text

def prepare_edgar_documents(
    df: pd.DataFrame,
    target_sections: Optional[list] = None,
    min_chars: int = EDGAR_MIN_SECTION_CHARS,
) -> pd.DataFrame:
    ...
    if target_sections is None:
        target_sections = EDGAR_TARGET_SECTIONS

    rows = []
    for filing_idx, row in df.iterrows():
        filename = row.get("filename", f"unknown_{filing_idx}")
        cik = row.get("cik", "")
        year = row.get("year", "")

        for section in target_sections:
            if section not in row:
                continue
            raw_text = row[section]
            cleaned = basic_clean(str(raw_text)) if raw_text is not None else ""

def prepare_earnings_documents(df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    dropped_too_short = 0  # NEW: counter for transparency

    for call_idx, row in df.iterrows():
        raw = row.get("transcript", "")
        if not isinstance(raw, str) or len(raw) < 1000:
            dropped_too_short += 1  # NEW: count drops
            continue

        cleaned = basic_clean(raw)
```

- Now what is the next step make any changes and re-run phase-2 or are we good to move forward to phase 3

reconcile_prepared_count (notebooks/03c_)
Here's an updated diagnostic that mirrors the real pipeline precisely, and then directly reconciles against the 60:


In [4]:
# notebooks/03c_reconcile_prepared_count.ipynb (single cell)
import sys
from pathlib import Path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from src.data_loading.loaders import load_earnings_sample
from src.preprocessing.cleaners import (
    basic_clean, strip_participant_header, split_transcript_on_qa,
)

raw = load_earnings_sample(sample_size=100, seed=42)
rows = []
for idx, r in raw.iterrows():
    # Mirror the REAL pipeline exactly — including preserve_newlines=True
    cleaned = basic_clean(r["transcript"], preserve_newlines=True)
    cleaned, header_stripped = strip_participant_header(cleaned)

    prepared, qa, was_split = split_transcript_on_qa(cleaned)

    rows.append({
        "idx": idx,
        "ticker": r.get("ticker"),
        "header_stripped": header_stripped,
        "was_split": was_split,
        "prepared_len": len(prepared) if was_split else None,
        "qa_len": len(qa) if was_split else None,
        # The two filters that decide whether a prepared unit survives:
        "prepared_survives": was_split and len(prepared) >= 500,
        # Bucket this call into the 500-char gap zone for inspection:
        "prepared_in_gap": was_split and 0 < len(prepared) < 500,
    })

diag = pd.DataFrame(rows)

print("Calls split (was_split=True):", diag["was_split"].sum())
print("Prepared units that survive (>=500):", diag["prepared_survives"].sum())
print("Prepared sections in the 1-499 gap (dropped):", diag["prepared_in_gap"].sum())
print()

# THE KEY TABLE: calls where prepared was dropped despite being split.
# These are the calls explaining the 71 -> 60 gap.
dropped = diag[diag["was_split"] & ~diag["prepared_survives"]]
print(f"Calls split but prepared dropped: {len(dropped)}")
print(dropped[["idx", "ticker", "header_stripped",
               "prepared_len", "qa_len"]].to_string())

Loading S&P 500 earnings transcripts split=train...
Full dataset size: 20681 transcripts
Sampled 100 transcripts
Calls split (was_split=True): 73
Prepared units that survive (>=500): 60
Prepared sections in the 1-499 gap (dropped): 13

Calls split but prepared dropped: 13
    idx ticker  header_stripped  prepared_len   qa_len
9     9   TROW            False         166.0  50956.0
15   15    ELV             True         101.0  37020.0
21   21    UNH             True          13.0  47576.0
39   39   TMUS             True          24.0  65247.0
40   40    CAT             True          13.0  38505.0
45   45   MDLZ             True          24.0  33280.0
51   51    ROP             True          13.0  27312.0
59   59    LMT             True          13.0  33462.0
60   60   INTC            False         277.0  45058.0
64   64    VST             True          24.0  34032.0
69   69    EOG             True          24.0  26413.0
92   92    MRK             True          13.0  36686.0
93   93    A